# 9c · Periodic energy — a boundary cue the aligner did not have

Notebook 9 builds TextGrids; 9b asks how much of them is measured rather than
ruled. **This one adds a measurement where there was only a rule.**

Every gap rule up to `vc-sil` decides a blank run from the CTC posteriors, the
phone classes, or a broadband spectral-change curve. None of them can tell
*noise* from *voicing*. So they place the edge of a fricative by the same logic
they use for a nasal, and three things stay broken:

* **VOT is invisible.** /t/ + /a/ is closure, burst, aspiration, *then* voicing.
  Spectral flux peaks at the **burst**, so the vowel was made to start there and
  the aspiration was scored as vowel.
* **Fricative–vowel edges are soft.** /s/ and /h/ are loud, so any
  amplitude-based cue puts the vowel too early.
* **A pause inside a word is not a pause.** Closure silence belongs to the stop
  that follows it, not to the phone in front.

## What periodic energy is

Following **ProPer — PROsodic analysis with PERiodic energy** (Aviad Albert,
Francesco Cangemi, Mark Ellison & Martine Grice, IfL Phonetik / SFB 1252,
University of Cologne, <https://osf.io/28ea5/>), we measure not how loud the
signal is but **how much of its energy is periodic**. Turbulent noise, however
loud, contributes nothing.

Two details are taken from that work rather than invented here:

* the zero of the scale is set by the signal itself — *"we measure purely
  voiceless portions and set the maximal periodic energy value of those
  voiceless portions as the constant floor denominator of the log variable"*
  (Albert et al. 2018:805), so a fricative sits at 0 dB **by construction**;
* boundaries are read off the curve's shape rather than a threshold — *"minima
  (and maxima) in the periodic energy function are indices of boundaries (and
  peaks) of periodic cycles"* (ibid.:807).

ProPer runs on Praat's periodicity and smooths with LOESS in R. This notebook
has neither, so periodicity comes from the **average magnitude difference
function**, the cheapest classical period detector:

$$\mathrm{AMDF}(\tau)=\frac{1}{N}\sum_n \left| x[n]-x[n+\tau] \right|$$

which collapses toward zero when $\tau$ is the period and stays near its average
for noise. That is less accurate than Praat for *F0* — we are not using it for
F0 — and quite good enough for *"is this frame periodic"*, which is all the
alignment rule asks.

In [ ]:
!pip -q install torch torchaudio transformers librosa soundfile pandas matplotlib scipy
import numpy as np, pandas as pd, torch
print("torch", torch.__version__, "·",
      "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
import librosa, matplotlib.pyplot as plt
from kolsch_align import (Aligner, MODES, VOCALIC, FRICATIVE, PLOSIVE,
                          AFFRICATE, NASAL, APPROXIMANT, SAMPLE_RATE,
                          coverage, silence_ms, read_textgrid_tier,
                          write_textgrid, word_positions, boundary_errors)
from kolsch_periodic import (periodic_power, periodicity, crossing, on_grid,
                             cue_strength)

MODEL_DIR     = os.environ.get("KOLSCH_MODEL",     os.path.join(MODELS, "kolsch_wav2vec2_model"))
PROCESSOR_DIR = os.environ.get("KOLSCH_PROCESSOR", MODEL_DIR)

al = Aligner(MODEL_DIR, PROCESSOR_DIR, lexicon=LEXICON)
print("modes:", ", ".join(MODES))

man = pd.read_csv(os.path.join(SEG, "manifest.csv"))
print(len(man), "segments")

# Voicing, not sonority, is the distinction this notebook is about, so the
# fricatives are split by it. /h/ is listed voiceless here and is the interesting
# case: BETWEEN VOWELS it is breathy voiced, and section 3 is about what the
# rule does when that happens.
VOICELESS_FRIC = set("f s ç ʃ χ h".split())
VOICED_FRIC    = FRICATIVE - VOICELESS_FRIC
print("voiceless fricatives:", " ".join(sorted(VOICELESS_FRIC)))
print("voiced fricatives:   ", " ".join(sorted(VOICED_FRIC)))

## 1 · Does this cue exist in **your** audio?

**Run this before anything else, because on the sample shipped with this repo
the answer is no.**

AMDF reads periodicity from the depth of a minimum, and broadband noise fills
that minimum in. On a quiet recording a vowel reaches a periodicity around 0.85
and a voiceless fricative stays near 0.3 — that contrast is what every rule
below depends on. Add 20 dB of hiss and the vowels drop to about 0.6 while the
fricatives do not move: the contrast is gone, and worse, the *floor* then gets
computed from frames that are not actually voiceless, so the whole curve
flattens toward zero.

This project has one corpus of each kind, which is why the check exists:

| | noise floor | periodicity when loud | vowel vs voiceless fricative |
|---|---|---|---|
| 84 field recordings, 16 kHz | −46 dB | 0.85 | d′ = **1.1** |
| 55 archival CD cuts *(shipped here)* | −11 to −27 dB | 0.6 | d′ = **0.08** |

The second row is the sample data below. So expect this section to tell you the
cue is absent, and expect section 4 to show `pe` changing very little as a
result. That is the honest outcome on this material, not a broken install.

In [ ]:
rows = []
for _, r in man.iterrows():
    wav, _ = librosa.load(r["audio_path"], sr=SAMPLE_RATE)
    t, pp, per, _ = periodic_power(wav, SAMPLE_RATE)
    if not len(t):
        continue
    ph, _, _ = al.align(r["audio_path"], r["text"], mode="vc")
    for p in ph:
        m = (t >= p["start"]) & (t < p["end"])
        if m.sum() < 2:
            continue
        cls = ("vowel"          if p["label"] in VOCALIC        else
               "nasal/liquid"   if p["label"] in NASAL | APPROXIMANT else
               "voiced fric."   if p["label"] in VOICED_FRIC    else
               "voiceless fric." if p["label"] in VOICELESS_FRIC else
               "plosive"        if p["label"] in PLOSIVE | AFFRICATE else "other")
        rows.append({"class": cls, "pp": pp[m].mean(), "per": per[m].mean()})

d = pd.DataFrame(rows)
print("mean periodic energy, dB over each recording's own voiceless floor\n")
print(d.groupby("class")["pp"].agg(["mean", "median", "count"])
       .sort_values("mean", ascending=False).round(1))

In [ ]:
diag = []
for _, r in man.iterrows():
    wav, _ = librosa.load(r["audio_path"], sr=SAMPLE_RATE)
    c = cue_strength(wav, SAMPLE_RATE)
    if "per_loud" in c:
        diag.append(c)
dg = pd.DataFrame(diag)
print(f"noise floor : median {dg['noise_floor_db'].median():6.1f} dB below peak")
print(f"periodicity : {dg['per_loud'].median():.2f} in the loudest frames, "
      f"{dg['per_quiet'].median():.2f} in the quietest")
print(f"usable      : {dg['usable'].sum()}/{len(dg)} recordings\n")
CUE_OK = dg["usable"].mean() >= 0.5
print("VERDICT:", "the cue is present — pe has something to measure" if CUE_OK
      else "the cue is NOT present in this audio — pe would read its landmarks\n"
           "         off noise. Use vc or vc-sil on material like this.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
order = (d.groupby("class")["pp"].mean().sort_values().index.tolist())
ax.boxplot([d.loc[d["class"] == c, "pp"] for c in order], vert=False,
           showfliers=False)
ax.set_yticks(range(1, len(order) + 1)); ax.set_yticklabels(order)
ax.set_xlabel("periodic energy, dB over the voiceless floor")
ax.set_title("If the boxes overlap completely, the cue is not there in your audio")
ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

v = d.loc[d["class"] == "vowel", "pp"]
f = d.loc[d["class"] == "voiceless fric.", "pp"]
if len(v) > 2 and len(f) > 2:
    dp = (v.mean() - f.mean()) / np.sqrt((v.var() + f.var()) / 2)
    print(f"vowel vs voiceless fricative: {v.mean():.1f} vs {f.mean():.1f} dB, "
          f"d' = {dp:.2f}")
    print("This agrees with the verdict above: d' near 0 means the rule below "
          "has\nnothing to work with, whatever it then does.")

## 2 · The rule

`--absorb pe` is `vc-sil` with **one** change: where a gap runs from an obstruent
to a sonorant, the boundary goes at **voicing onset** — the halfway crossing of
the periodic-energy rise across that gap. After a stop that is the end of VOT,
so closure, burst and aspiration stay inside the consonant. After /s/ or /ʃ/ it
is where the vowel begins rather than where the noise gets loud.

Everything else is left exactly where `vc-sil` put it. Three plausible
extensions were tried and all three lost; they are listed in section 5.

In [ ]:
import inspect
from kolsch_align import absorb_gaps_pe
print(inspect.getsource(absorb_gaps_pe))

## 3 · The case that decides how far to trust it

**A voiced obstruent has no voicing onset.** Intervocalic /h/ in German and
Kölsch is breathy *voiced*: periodic energy never approaches the floor, so there
is no rise to find, and a crossing computed on a curve that merely wobbles is an
invented landmark, not a measurement.

So the rule asks first whether there is anything to find — `aperiodic_db`, the
gate. Where the curve never nears the floor, the blank goes to the sonorant
instead, which is where the CTC spike already said the obstruent ended.

The cell below finds both kinds of gap in your own audio and reports how often
each occurs. **If almost every gap takes the gate, the periodic landmark is
doing nothing for your material** and `vc-sil` is the honest choice.

In [ ]:
APERIODIC_DB = 6.0          # absorb_gaps_pe's default gate
SONORANT = VOCALIC | NASAL | APPROXIMANT

fired, gated = [], []
for _, r in man.iterrows():
    wav, _ = librosa.load(r["audio_path"], sr=SAMPLE_RATE)
    t, pp, _, _ = periodic_power(wav, SAMPLE_RATE)
    raw, _, _ = al.align(r["audio_path"], r["text"], mode="none")   # CTC spikes
    for a, b in zip(raw, raw[1:]):
        if b["start"] <= a["end"]:
            continue
        if not (b["label"] in SONORANT and a["label"] not in SONORANT):
            continue
        lo, hi = np.searchsorted(t, a["end"]), np.searchsorted(t, b["start"])
        if hi - lo < 2:
            continue
        rec = {"phones": f'{a["label"]}→{b["label"]}',
               "gap_ms": (b["start"] - a["end"]) * 1000,
               "floor_db": pp[lo:hi].min()}
        (gated if rec["floor_db"] >= APERIODIC_DB else fired).append(rec)

n = len(fired) + len(gated)
print(f"{n} obstruent→sonorant gaps")
print(f"  {len(fired):4d} reach the floor  → cut at voicing onset")
print(f"  {len(gated):4d} never do         → gate taken, blank to the sonorant")
if gated:
    print("\nmost common gated (voiced obstruent, nothing to find):")
    print(pd.DataFrame(gated)["phones"].value_counts().head(6).to_string())
if not CUE_OK:
    print("\n!! Section 1 said the cue is absent here, so read these counts as "
          "a\n   demonstration of the mechanism, not as evidence about these "
          "phones.\n   On a flat curve almost everything 'reaches the floor' "
          "because the whole\n   curve is at the floor.")

## 4 · What it changes, and where

Durations first: the rule should lengthen obstruents at the expense of the
sonorant after them, because it stops handing aspiration and frication to the
vowel. Any *other* phone that moves is a bug.

In [ ]:
row = man.iloc[0]
print(row["text"], "\n")
runs = {m: al.align(row["audio_path"], row["text"], mode=m)[0]
        for m in ("hybrid", "vc", "vc-sil", "pe")}

tbl = pd.DataFrame({"phone": [p["label"] for p in runs["pe"]]})
tbl["class"] = ["sonorant" if p in SONORANT else "obstruent" for p in tbl["phone"]]
for m in runs:
    tbl[m] = [round((p["end"] - p["start"]) * 1000) for p in runs[m]]
tbl["pe−vc-sil"] = tbl["pe"] - tbl["vc-sil"]
display(tbl)

moved = tbl[tbl["pe−vc-sil"] != 0]
print(f"{len(moved)} of {len(tbl)} phones differ from vc-sil at all.")
print("Everything unchanged is unchanged BY CONSTRUCTION — pe overrides "
      "specific boundaries\nand copies the rest, so an identical row is a "
      "guarantee rather than a coincidence.")

In [ ]:
tot = {m: [] for m in runs}
for _, r in man.iterrows():
    for m in runs:
        ph, _, _ = al.align(r["audio_path"], r["text"], mode=m)
        for p in ph:
            tot[m].append((p["label"], (p["end"] - p["start"]) * 1000))

base = dict()
print("mean duration change vs vc-sil over the whole sample, ms\n")
print(f"{'':16}{'obstruents':>12}{'sonorants':>12}")
ref = {i: v for i, v in enumerate(tot["vc-sil"])}
for m in ("hybrid", "vc", "pe"):
    ob = [v - ref[i][1] for i, (lab, v) in enumerate(tot[m])
          if lab not in SONORANT]
    so = [v - ref[i][1] for i, (lab, v) in enumerate(tot[m])
          if lab in SONORANT]
    print(f"{m:16}{np.mean(ob):+12.1f}{np.mean(so):+12.1f}")
print("\npe should be POSITIVE for obstruents and negative for sonorants: it is "
      "giving\nfrication and aspiration back to the consonant they belong to.")

## 5 · Against another aligner

Everything above is internal consistency. To know whether the boundaries are
*better* you need a second opinion, and one you did not derive from this model.

Drop TextGrids from MFA, MAUS, or a human annotator into
`data/reference_textgrids/` — same file stems, same phone chain, index for index
— and this section scores against them. Without it the section skips.

**On the project's own material** (84 field recordings, 941 word boundaries and
2717 word-internal ones, scored against MFA and MAUS *separately* because both
descend from HMM-GMM and drifting toward one drifts toward the other):

| | vc-sil | **pe** |
|---|---|---|
| MFA · word-initial onset | 40.0 | **37.5** |
| MFA · word-internal onset | 26.0 | **22.7** |
| MFA · word-final end | 50.0 | **47.5** |
| MAUS · word-initial onset | 42.9 | 42.9 |
| MAUS · word-internal onset | 34.3 | **32.1** |
| MAUS · word-final end | 52.9 | 52.9 |

Never worse on any bucket against either reference, and the gain concentrates
where it was designed to: **voiceless fricative → vowel goes 30.4 → 21.6 ms
against MFA and 39.0 → 33.5 against MAUS.**

### Four things that sounded just as good and were not

Each was implemented, measured on the same 3658 boundaries, and rejected:

| variant | what it did | why it lost |
|---|---|---|
| `offset=True` | mirror rule: voicing **offset** at sonorant→obstruent | costs 9 ms at nasal–stop edges against both references. Voicing onset is abrupt; voicing offset is gradual — devoicing, creak — so "where the curve falls" is not a location. |
| `onset="rise"` | ProPer's own landmark, the **steepest** rise | lands late into the vowel: 22.5 ms from MFA at stop–vowel edges vs 16.6 for the halfway crossing. |
| `cross_only=False` | landmarks at **every** gap | between two sonorants the curve is flat, and a landmark read off a flat curve is noise: 12 ms worse at nasal–vowel and V–V. |
| `aperiodic_db=0` | trust the crossing even on **voiced** obstruents | the case in section 3. Fvl→V collapses from 21.6 to 40.2 ms against MFA. |

### And one honest confound

`pe`'s gain at **voiced** fricative→vowel is *not* the periodic curve. Dropping
one unrelated clause of the `vc` rule — its exception for fricatives between two
vowels — gets 20.4 / 18.2 ms there with no curve at all, beating `pe`'s
20.1 / 17.7 by nothing worth claiming. The curve earns its place at
**voiceless** fricatives, where dropping that clause instead makes things
*worse* (35.1 / 48.3). Two effects, two different sets of phones; the control
column in `12_eval_periodic.py` is what separates them.

In [ ]:
REF_DIR = os.path.join(DATA, "reference_textgrids")
refs = sorted(Path(REF_DIR).glob("*.TextGrid")) if os.path.isdir(REF_DIR) else []
if not refs:
    print(f"No reference TextGrids in {REF_DIR} — section 5 skipped.")
    print("The table above reports what this rule scored on the project's own "
          "material.")
else:
    out = []
    for tg in refs:
        hit = man[man["audio_path"].apply(lambda p: Path(p).stem == tg.stem)]
        if hit.empty:
            continue
        r = hit.iloc[0]
        ref = read_textgrid_tier(tg, "phones")
        chain, _ = al.text_to_chain(r["text"])
        for m in ("vc", "vc-sil", "pe"):
            ph, _, _ = al.align(r["audio_path"], r["text"], mode=m)
            e = boundary_errors(ph, ref, chain)
            for k, v in e.items():
                out.append({"file": tg.stem, "mode": m, "bucket": k, "ms": v})
    if out:
        piv = (pd.DataFrame(out).pivot_table(index="bucket", columns="mode",
                                             values="ms", aggfunc="median")
               .round(1))
        display(piv)
        print("median |Δ| in ms. Check the phone chains line up index for "
              "index,\notherwise this compares two dictionaries and not two "
              "aligners.")
    else:
        print("No reference file matched a manifest stem.")

## 6 · Export

`vc` remains the documented default of this repo and `pe` is the better rule on
the evidence above — but that evidence is MFA and MAUS, and they are not ground
truth. Against the only **human-placed** boundaries in this project, three
hand-cut excerpt edges, `vc` is 7.8 ms out and both `vc-sil` and `pe` are 36.0.
`pe` inherits that from `vc-sil` and changes none of those three edges, so this
notebook does not resolve that disagreement — it leaves it exactly where 9b did.

Choose on what you need — **and only after section 1 said the cue is there**:

* **`vc`** — phones tile the signal end to end, no holes. Closest to the human
  edges we have. Use it if downstream code assumes a gapless tier, and use it on
  noisy material regardless of anything else on this page.
* **`vc-sil`** — as `vc`, but pauses between words become holes.
* **`pe`** — `vc-sil` plus voicing-onset edges. Best against MFA and MAUS on
  quiet recordings. Use it for duration measurements on obstruents. On audio
  that fails the section 1 check it is strictly worse than `vc-sil`, because it
  moves boundaries on the strength of a curve that is measuring hiss.

The cell below picks for you from what section 1 measured.

In [ ]:
ABSORB = "pe" if CUE_OK else "vc"       # section 1 decides; override if you like
print(f"exporting in mode {ABSORB!r}"
      + ("" if CUE_OK else "  — pe withheld: this audio is too noisy for it"))
TG_DIR = os.path.join(DATA, "textgrids"); os.makedirs(TG_DIR, exist_ok=True)

ok = 0
for _, r in man.iterrows():
    stem = Path(r["audio_path"]).stem
    try:
        ph, wd, dur = al.align(r["audio_path"], r["text"], mode=ABSORB)
        write_textgrid(os.path.join(TG_DIR, f"{stem}.TextGrid"), dur,
                       [("words", wd), ("phones", ph)])
        ok += 1
    except Exception as e:
        print(f"  {stem}: {type(e).__name__}: {e}")
print(f"{ok} TextGrids in mode {ABSORB!r} -> {TG_DIR}")

---

**Albert, A., Cangemi, F., Ellison, T. M. & Grice, M.** *ProPer: PROsodic
analysis with PERiodic energy.* <https://osf.io/28ea5/> ·
**Albert, A., Cangemi, F. & Grice, M.** (2018) Using periodic energy to enrich
acoustic representations of pitch in speech: a demonstration. *Speech Prosody
2018*, 804–807. doi:10.21437/SpeechProsody.2018-162

The implementation here is not ProPer: it is a Praat-free, R-free
reimplementation of two of its ideas (the voiceless floor, and reading
boundaries off the curve's shape) using AMDF for periodicity. Errors in it are
ours, not theirs.